# 08 - PySpark, SparkSQL & Table Management

**Objective:** Introduce Apache Spark via PySpark — DataFrame transformations,
SparkSQL queries, table/metadata management, and a brief Spark MLlib comparison,
all using the Bike Sharing dataset.

**Concepts Covered:**
- SparkSession setup and PySpark DataFrame basics
- DataFrame transformations: `select`, `filter`, `groupBy`, `agg`, `withColumn`
- SparkSQL: temporary views, SQL queries, aggregations, JOINs, window functions, UDFs
- Table management: Parquet I/O, managed vs unmanaged tables, partitioning, metadata inspection
- Brief Spark MLlib Pipeline (VectorAssembler → StandardScaler → LogisticRegression)

> **Note:** PySpark 4.1.1 is used. The dataset is small, making it ideal for learning
> the PySpark API without cluster overhead.


In [ ]:
# TODO: Set up SparkSession and import libraries
#
# from pathlib import Path
# import pandas as pd
# import numpy as np
#
# from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType
# from pyspark.sql.functions import col, mean, stddev, when, udf
# from pyspark.sql.window import Window
#
# from pyspark.ml.feature import VectorAssembler, StandardScaler
# from pyspark.ml.classification import LogisticRegression
# from pyspark.ml import Pipeline
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
#
# import warnings
# warnings.filterwarnings("ignore")
#
# spark = SparkSession.builder \
#     .appName("bike_sharing_spark") \
#     .master("local[*]") \
#     .config("spark.sql.adaptive.enabled", "false") \
#     .getOrCreate()
#
# print("Libraries imported successfully")
# print(f"Spark version: {spark.version}")


## 1. PySpark DataFrame Basics

PySpark's core abstraction is the **DataFrame** — a distributed collection of
rows with named columns. Unlike pandas, operations are **lazy**: transformations
are queued and only executed when an action (`.show()`, `.count()`, `.collect()`)
is called.


In [ ]:
# TODO: Read the clean data as a PySpark DataFrame
#
# processed_dir = Path("../data/processed")
# df = spark.read \
#     .option("header", True) \
#     .option("inferSchema", True) \
#     .csv(str(processed_dir / "clean_data.csv"))
#
# print(f"Rows: {df.count()}, Columns: {len(df.columns)}")
# df.printSchema()


In [ ]:
# TODO: Basic DataFrame operations
#
# Inspect first 5 rows with df.show()
# Cast target column to integer: df = df.withColumn("target", col("target").cast("int"))
# Group by target and compute mean/stddev of key features


In [ ]:
# TODO: Filter and select operations
#
# Filter for positive class (target == 1)
# Select specific columns + create a computed ratio column


## 2. SparkSQL

SparkSQL allows you to query DataFrames using standard SQL. Register a
DataFrame as a **temporary view**, then run `spark.sql()`.


In [ ]:
# TODO: Register a temp view and run SQL queries
#
# df.createOrReplaceTempView("bike_sharing")
#
# Basic SQL: SELECT, GROUP BY, AVG, ORDER BY
# Complex SQL: WHERE, GROUP BY, HAVING


In [ ]:
# TODO: Create multiple views and JOIN them
#
# Split DataFrame into features view and labels view
# Add a row identifier with monotonically_increasing_id()
# JOIN the views and aggregate


In [ ]:
# TODO: Window functions
#
# Use Window.partitionBy().orderBy() with rank()
# Show top-N rows per partition


In [ ]:
# TODO: Python UDF with SparkSQL
#
# Define a function that maps a numeric feature to a category
# Register it as a UDF
# Use it in a SparkSQL query


## 3. Table Management

Spark supports **managed tables** (stored in the Spark warehouse directory,
lifecycle managed by Spark) and **unmanaged tables/external tables** (backed by
user-specified file paths). Parquet is the default storage format.


In [ ]:
# TODO: Save as Parquet and read back
#
# df.write.mode("overwrite").parquet(str(processed_dir / "data_spark.parquet"))
# df_parquet = spark.read.parquet(...)
# print(f"Parquet rows: {df_parquet.count()}")


In [ ]:
# TODO: Create and inspect a managed table
#
# spark.sql("DROP TABLE IF EXISTS bike_sharing")
# df.write.mode("overwrite").saveAsTable("bike_sharing")
# spark.sql("SHOW TABLES").show()
# spark.sql(f"DESCRIBE EXTENDED {table_name}").show()


In [ ]:
# TODO: Partitioning by target
#
# df.write.mode("overwrite").partitionBy("Rented Bike Count").parquet(str(processed_dir / "data_partitioned"))
# Read partition and filter to show partition pruning


In [ ]:
# TODO: Clean up — drop managed table
#
# spark.sql("DROP TABLE IF EXISTS bike_sharing")
# spark.sql("SHOW TABLES").show()


## 4. Brief Spark MLlib Pipeline Comparison

Spark MLlib provides a Pipeline API similar to sklearn. Here we build
a `VectorAssembler` → `StandardScaler` → `LogisticRegression` pipeline
and compare AUC with the sklearn model from Notebook 04.


In [ ]:
# TODO: Prepare features for MLlib
#
# feature_cols = [c for c in df.columns if c != "Rented Bike Count"]
# df_ml = df.select("{target}", *feature_cols).dropna()
# print(f"Using {len(feature_cols)} features, {df_ml.count()} rows")


In [ ]:
# TODO: Build and train a Spark MLlib Pipeline
#
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
# scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
# lr = LogisticRegression(featuresCol="features", labelCol="Rented Bike Count")
# spark_pipeline = Pipeline(stages=[assembler, scaler, lr])
#
# train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
# spark_model = spark_pipeline.fit(train)
# predictions = spark_model.transform(test)
#
# evaluator = BinaryClassificationEvaluator(labelCol="{target}", metricName="areaUnderROC")
# spark_auc = evaluator.evaluate(predictions)
# print(f"Spark LogisticRegression AUC: {spark_auc:.4f}")


In [ ]:
# TODO: Stop SparkSession
#
# spark.stop()
# print("SparkSession stopped.")


## Summary

- **PySpark DataFrame API**: lazy transformations, actions, filtering, aggregation
- **SparkSQL**: temporary views, SQL queries, JOINs, window functions, UDFs
- **Table Management**: Parquet I/O, managed vs unmanaged tables, partitioning, metadata
- **Spark MLlib**: Pipeline API with VectorAssembler, StandardScaler, LogisticRegression


### Exercises

1. **SQL feature ranking**: Write a query that computes a ratio of two features.
2. **Managed table with partitioning**: Create, verify partitions, drop.
3. **UDF for staging**: Register a UDF for severity/category assignment.
4. **Parquet vs CSV comparison**: Compare file sizes and read times.
5. **Spark MLlib with RandomForest**: Replace LogisticRegression and compare AUC.
6. **Repartitioning experiment**: Try `df.repartition(4)` vs `df.coalesce(1)`.
